# phys119-functions: verify an install

Run this **from the account being tested**, in a folder that does not
contain a copy of `phys119_functions.py`. Run every cell in order. The last
cell prints a single pass or fail verdict.

Two settings in the first cell:

- `EXPECTED` is the version to expect. Change it for a later release.
- `EXPECT_LOCATION` is where the package should be coming from:
  - `"shared"` after a system-wide install into `/opt/tljh/user`, which is
    what every student gets;
  - `"user"` after an install into one account only
    (`~/.local/lib/python3.10/site-packages`), which is the staged test of a
    single student account before going server-wide.

In [1]:
EXPECTED = "1.1.0"
EXPECT_LOCATION = "shared"   # "shared" or "user"

import os
import site
import sys

import phys119_functions as pf

module_path = os.path.realpath(pf.__file__)
shared_prefix = os.path.realpath(sys.prefix)
user_site = os.path.realpath(site.getusersitepackages())

if module_path.startswith(user_site):
    location = "user"
elif module_path.startswith(shared_prefix):
    location = "shared"
elif os.path.dirname(module_path) == os.path.realpath(os.getcwd()):
    location = "working directory"
else:
    location = "somewhere else"

print("Account:           ", os.path.expanduser("~"))
print("Kernel executable: ", sys.executable)
print("Shared environment:", shared_prefix)
print("Module loaded from:", module_path)
print("That location is:  ", location, "(expected: " + EXPECT_LOCATION + ")")
print("Version:           ", getattr(pf, "__version__", "NONE"))
print()

problems = []

if getattr(pf, "__version__", None) != EXPECTED:
    problems.append(
        "Version is " + str(getattr(pf, "__version__", "missing"))
        + ", expected " + EXPECTED
    )

if location == EXPECT_LOCATION:
    print("OK: loaded from the", location, "location.")
elif location == "working directory":
    problems.append(
        "Loaded from the working directory. A stray phys119_functions.py "
        "sits next to this notebook."
    )
elif location == "user" and EXPECT_LOCATION == "shared":
    problems.append(
        "Loaded from this account's personal packages, which is hiding the "
        "shared install. Run: pip uninstall --user phys119-functions"
    )
elif location == "shared" and EXPECT_LOCATION == "user":
    problems.append(
        "Loaded from the shared environment, so the per-account install "
        "under test is not the copy being used."
    )
else:
    problems.append("Loaded from an unexpected place: " + module_path)

Account:            C:\Users\josse
Kernel executable:  C:\tmp\nbenv\Scripts\python.exe
Shared environment: C:\tmp\nbenv
Module loaded from: C:\tmp\nbenv\Lib\site-packages\phys119_functions.py
That location is:   shared (expected: shared)
Version:            1.1.0

OK: loaded from the shared location.


## Values

Checked against numpy, so this catches a broken install rather than a
mistyped constant.

In [2]:
import numpy as np

from phys119_functions import *

sample = [10.1, 10.3, 9.8]

checks = [
    ("t_score(10, 2, 12, 3)", t_score(10, 2, 12, 3), 0.5547001962252291),
    ("standard_deviation(sample)", standard_deviation(sample), float(np.std(sample, ddof=1))),
    ("standard_unc_of_mean(sample)", standard_unc_of_mean(sample),
     float(np.std(sample, ddof=1) / np.sqrt(len(sample)))),
]

for label, got, want in checks:
    ok = got is not None and abs(got - want) < 1e-12 and type(got) is float
    print(("OK  " if ok else "FAIL"), label, "->", got)
    if not ok:
        problems.append(label + " gave " + repr(got) + ", expected " + repr(want))

OK   t_score(10, 2, 12, 3) -> 0.5547001962252291
OK   standard_deviation(sample) -> 0.25166114784235827
OK   standard_unc_of_mean(sample) -> 0.14529663145135577


## What `import *` brings in

Expect the three function names plus `Phys119Error`, and nothing else.

In [3]:
namespace = {}
exec("from phys119_functions import *", namespace)
exported = sorted(name for name in namespace if not name.startswith("__"))

print(exported)
if exported != ["Phys119Error", "standard_deviation", "standard_unc_of_mean", "t_score"]:
    problems.append("import * exported " + repr(exported))

['Phys119Error', 'standard_deviation', 'standard_unc_of_mean', 't_score']


## Error behaviour

The next cell should show a red box about needing at least 2 measurements,
with **no traceback**. The `try` is here only so this notebook reaches its
verdict: an error stops the cell, so without it nothing below would run.

In [4]:
try:
    standard_deviation([4.2])
    problems.append("an impossible call did not stop the cell")
except Phys119Error:
    pass

## Verdict

In [5]:
print()
if problems:
    print("FAILED")
    for problem in problems:
        print(" -", problem)
else:
    scope = "for all users" if EXPECT_LOCATION == "shared" else "for this account"
    print("ALL CHECKS PASSED")
    print("phys119-functions", EXPECTED, "is installed", scope, "and behaves correctly.")


ALL CHECKS PASSED
phys119-functions 1.1.0 is installed for all users and behaves correctly.
